<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/Random_Nonproxy_Ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_rel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from fairlearn.metrics import (
    demographic_parity_difference,
    equalized_odds_difference,
    demographic_parity_ratio,
    equalized_odds_ratio,
)
import warnings
warnings.filterwarnings('ignore')


# ==========================================
# HELPER: SSL-FREE DATA LOADING
# ==========================================

def load_adult() -> pd.DataFrame:
    """Load UCI Adult dataset with SSL bypass."""
    print("\n[Loading UCI Adult Dataset]")

    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income",
    ]

    # Try local file first
    try:
        df = pd.read_csv(
            "adult.csv",
            names=columns,
            skipinitialspace=True,
            na_values=["?"],
        )
        print("  ✓ Loaded from local file: adult.csv")
    except FileNotFoundError:
        # Try downloading
        import ssl
        from io import StringIO
        import urllib.request

        url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
        print(f"  Downloading from: {url}")

        try:
            context = ssl._create_unverified_context()
            with urllib.request.urlopen(url, context=context) as response:
                data = response.read().decode('utf-8')
            df = pd.read_csv(
                StringIO(data),
                names=columns,
                skipinitialspace=True,
                na_values=["?"],
            )
            print("  ✓ Downloaded successfully")
        except Exception as e:
            print(f"  ✗ Download failed: {str(e)[:50]}")
            raise

    print(f"  Original shape: {df.shape}")

    # Preprocess
    df = df.dropna().copy()
    print(f"  After dropping NaN: {df.shape}")

    df = df.drop(columns=["fnlwgt", "education", "native-country"], errors='ignore')

    df["income"] = (df["income"].str.strip() == ">50K").astype(int)

    print(f"  Final shape: {df.shape}")

    return df


def load_acsincome(sample_size: int = 50000) -> pd.DataFrame:
    """Load ACSIncome dataset from folktables."""
    print("\n[Loading ACSIncome Dataset]")

    try:
        from folktables import ACSDataSource, ACSIncome
    except ImportError:
        raise ImportError("Please install folktables: pip install folktables")

    data_source = ACSDataSource(
        survey_year="2018",
        horizon="1-Year",
        survey="person",
    )

    print("  Downloading ACS data for CA...")
    acs_data = data_source.get_data(states=["CA"], download=True)
    print(f"  Downloaded shape: {acs_data.shape}")

    features, labels, groups = ACSIncome.df_to_numpy(acs_data)
    feature_names = ACSIncome.features

    df = pd.DataFrame(features, columns=feature_names)
    df["income"] = labels
    df["sex"] = groups

    print(f"  Original shape: {df.shape}")

    df = df.dropna()
    print(f"  After dropping NaN: {df.shape}")

    if sample_size is not None and len(df) > sample_size:
        print(f"  Sampling {sample_size} records...")
        df = df.sample(n=sample_size, random_state=42)
        print(f"  Sampled shape: {df.shape}")

    return df


# ==========================================
# MODEL PIPELINE BUILDER
# ==========================================

def build_model_pipeline(
    feature_columns: list,
    continuous_columns: list = None,
    model_type: str = "logistic",
) -> Pipeline:
    """Build preprocessing + model pipeline."""

    if continuous_columns is None:
        continuous_columns = []

    cont_cols = [c for c in feature_columns if c in continuous_columns]
    cat_cols = [c for c in feature_columns if c not in continuous_columns]

    transformers = []

    if cont_cols:
        transformers.append(
            (
                "cont",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                cont_cols,
            )
        )

    if cat_cols:
        try:
            # Try new sklearn API
            transformers.append(
                (
                    "cat",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False, max_categories=50)),
                        ]
                    ),
                    cat_cols,
                )
            )
        except TypeError:
            # Fall back to old API
            transformers.append(
                (
                    "cat",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="most_frequent")),
                            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False, max_categories=50)),
                        ]
                    ),
                    cat_cols,
                )
            )

    preprocess = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    if model_type == "logistic":
        model = LogisticRegression(
            max_iter=1000,
            solver="lbfgs",
            random_state=42,
            n_jobs=-1,
            tol=0.01,
        )
    elif model_type == "random_forest":
        model = RandomForestClassifier(
            n_estimators=50,
            max_depth=10,
            random_state=42,
            n_jobs=-1,
            min_samples_leaf=10,
        )
    elif model_type == "gradient_boosting":
        model = GradientBoostingClassifier(
            n_estimators=50,
            max_depth=4,
            learning_rate=0.05,
            random_state=42,
            subsample=0.8,
            min_samples_leaf=5,
            verbose=0,
        )
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    return Pipeline(
        steps=[
            ("preprocess", preprocess),
            ("model", model),
        ]
    )


# ==========================================
# FAIRNESS METRICS
# ==========================================

def compute_fairness_metrics(y_true: np.ndarray, y_pred: np.ndarray, s: np.ndarray) -> dict:
    """Compute fairness metrics."""

    metrics = {}

    try:
        metrics["dpd"] = float(demographic_parity_difference(
            y_true=y_true.astype(int),
            y_pred=y_pred.astype(int),
            sensitive_features=s.astype(int),
        ))
    except:
        metrics["dpd"] = np.nan

    try:
        metrics["dpr"] = float(demographic_parity_ratio(
            y_true=y_true.astype(int),
            y_pred=y_pred.astype(int),
            sensitive_features=s.astype(int),
        ))
    except:
        metrics["dpr"] = np.nan

    try:
        metrics["eod"] = float(equalized_odds_difference(
            y_true=y_true.astype(int),
            y_pred=y_pred.astype(int),
            sensitive_features=s.astype(int),
        ))
    except:
        metrics["eod"] = np.nan

    try:
        metrics["eor"] = float(equalized_odds_ratio(
            y_true=y_true.astype(int),
            y_pred=y_pred.astype(int),
            sensitive_features=s.astype(int),
        ))
    except:
        metrics["eor"] = np.nan

    return metrics


# ==========================================
# ABLATION EVALUATION
# ==========================================

def evaluate_dataset_ablation(
    data: pd.DataFrame,
    target: str,
    sensitive_attr: str,
    mb_features: list,
    proxy_features: list,
    continuous_columns: list = None,
    dataset_name: str = "",
    n_splits: int = 30,
    n_random_draws: int = 30,
    model_types: list = None,
) -> tuple:
    """
    Evaluate ablation: MB_Shield vs Fair_Filter vs Random_Filter.

    Returns:
        (split_results_df, summary_df, differences_df)
    """

    if model_types is None:
        model_types = ["logistic", "random_forest", "gradient_boosting"]

    if continuous_columns is None:
        continuous_columns = []

    print(f"\n[Ablation Evaluation for {dataset_name}]")
    print(f"  MB features: {mb_features}")
    print(f"  Proxy features: {proxy_features}")

    # Filter to available columns
    mb_features = [f for f in mb_features if f in data.columns]
    proxy_features = [f for f in proxy_features if f in proxy_features if f in data.columns]

    # Create feature sets
    mb_shield = sorted(set(mb_features) | {sensitive_attr})
    fair_filter = sorted(set(mb_features) - set(proxy_features))

    # For random filter: identify non-proxy MB features
    non_proxy_mb = [f for f in mb_features if f not in proxy_features]
    n_proxy = len(proxy_features)
    n_nonproxy = len(non_proxy_mb)
    n_random_to_remove = min(n_proxy, n_nonproxy)

    if n_random_to_remove < n_proxy:
        print(f"  Warning: Only {n_random_to_remove} non-proxy features available to remove")
        print(f"           (proxy features: {n_proxy}, non-proxy features: {n_nonproxy})")

    print(f"  MB_Shield: {len(mb_shield)} features")
    print(f"  Fair_Filter: {len(fair_filter)} features")
    print(f"  Random_Filter: {len(mb_shield)} features minus {n_random_to_remove} random non-proxies")

    # Prepare data
    X = data.drop(columns=[target, sensitive_attr])
    y = data[target].to_numpy()
    s = data[sensitive_attr].to_numpy()

    if y.dtype == bool:
        y = y.astype(int)
    if not np.issubdtype(s.dtype, np.integer):
        le = LabelEncoder()
        s = le.fit_transform(s.astype(str))

    splitter = StratifiedShuffleSplit(
        n_splits=n_splits,
        test_size=0.30,
        random_state=42,
    )

    split_rows = []
    total_iterations = n_splits * len(model_types) * (2 + n_random_draws)

    iteration = 0

    for split_idx, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
        print(f"\n  [Split {split_idx}/{n_splits}]")

        X_train_full = X.iloc[train_idx].copy()
        X_test_full = X.iloc[test_idx].copy()
        y_train = y[train_idx]
        y_test = y[test_idx]
        s_test = s[test_idx]

        for model_type in model_types:
            # ==========================================
            # 1. MB_SHIELD
            # ==========================================
            config_name = "MB_Shield"
            features_valid = [f for f in mb_shield if f in X.columns]

            iteration += 1
            print(f"    {config_name:20} {model_type:20} [{iteration}/{total_iterations}]", end=' ', flush=True)

            try:
                pipe = build_model_pipeline(features_valid, continuous_columns, model_type)
                X_train = X_train_full[features_valid]
                X_test = X_test_full[features_valid]

                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                fair_metrics = compute_fairness_metrics(y_test, y_pred, s_test)

                split_rows.append({
                    "dataset": dataset_name,
                    "split": split_idx,
                    "random_draw": np.nan,
                    "config": config_name,
                    "model": model_type,
                    "n_features": len(features_valid),
                    "features_used": tuple(features_valid),
                    "removed_features": tuple(),
                    "proxy_removed_count": 0,
                    "random_removed_count": 0,
                    "accuracy": acc,
                    "f1": f1,
                    **fair_metrics,
                })
                print("✓")
            except Exception as e:
                print(f"✗ {str(e)[:30]}")

            # ==========================================
            # 2. FAIR_FILTER
            # ==========================================
            config_name = "Fair_Filter"
            features_valid = [f for f in fair_filter if f in X.columns]
            removed_features = tuple(set(mb_shield) - set(features_valid))

            iteration += 1
            print(f"    {config_name:20} {model_type:20} [{iteration}/{total_iterations}]", end=' ', flush=True)

            try:
                pipe = build_model_pipeline(features_valid, continuous_columns, model_type)
                X_train = X_train_full[features_valid]
                X_test = X_test_full[features_valid]

                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)

                acc = accuracy_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                fair_metrics = compute_fairness_metrics(y_test, y_pred, s_test)

                split_rows.append({
                    "dataset": dataset_name,
                    "split": split_idx,
                    "random_draw": np.nan,
                    "config": config_name,
                    "model": model_type,
                    "n_features": len(features_valid),
                    "features_used": tuple(features_valid),
                    "removed_features": removed_features,
                    "proxy_removed_count": n_proxy,
                    "random_removed_count": 0,
                    "accuracy": acc,
                    "f1": f1,
                    **fair_metrics,
                })
                print("✓")
            except Exception as e:
                print(f"✗ {str(e)[:30]}")

            # ==========================================
            # 3. RANDOM_FILTER (multiple draws)
            # ==========================================
            config_name = "Random_Filter"

            for draw_idx in range(n_random_draws):
                iteration += 1
                print(f"    {config_name:20} {model_type:20} draw {draw_idx+1:2d} [{iteration}/{total_iterations}]", end=' ', flush=True)

                # Random seed for reproducibility
                rng = np.random.RandomState(42 + split_idx * 1000 + draw_idx)

                # Randomly select non-proxy features to remove
                if n_random_to_remove > 0:
                    removed_random = rng.choice(non_proxy_mb, size=n_random_to_remove, replace=False)
                else:
                    removed_random = []

                random_filter_features = [f for f in mb_shield if f not in removed_random]
                features_valid = [f for f in random_filter_features if f in X.columns]

                try:
                    pipe = build_model_pipeline(features_valid, continuous_columns, model_type)
                    X_train = X_train_full[features_valid]
                    X_test = X_test_full[features_valid]

                    pipe.fit(X_train, y_train)
                    y_pred = pipe.predict(X_test)

                    acc = accuracy_score(y_test, y_pred)
                    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
                    fair_metrics = compute_fairness_metrics(y_test, y_pred, s_test)

                    split_rows.append({
                        "dataset": dataset_name,
                        "split": split_idx,
                        "random_draw": draw_idx,
                        "config": config_name,
                        "model": model_type,
                        "n_features": len(features_valid),
                        "features_used": tuple(features_valid),
                        "removed_features": tuple(removed_random),
                        "proxy_removed_count": 0,
                        "random_removed_count": n_random_to_remove,
                        "accuracy": acc,
                        "f1": f1,
                        **fair_metrics,
                    })
                    print("✓")
                except Exception as e:
                    print(f"✗ {str(e)[:30]}")

    split_results = pd.DataFrame(split_rows)

    print(f"\n✓ Completed {len(split_results)} configurations")

    return split_results


# ==========================================
# SUMMARY COMPUTATION
# ==========================================

def compute_summary(split_results: pd.DataFrame) -> pd.DataFrame:
    """Compute summary statistics."""

    metrics = ["accuracy", "f1", "dpd", "dpr", "eod", "eor"]

    rows = []

    for dataset in split_results["dataset"].unique():
        for model in split_results["model"].unique():
            subset = split_results[
                (split_results["dataset"] == dataset) &
                (split_results["model"] == model)
            ]

            for metric in metrics:
                for config in ["MB_Shield", "Fair_Filter", "Random_Filter"]:
                    config_data = subset[subset["config"] == config]

                    if config_data.empty:
                        continue

                    if config == "Random_Filter":
                        # Average over random draws within each split first
                        split_means = []
                        for split_idx in config_data["split"].unique():
                            split_subset = config_data[config_data["split"] == split_idx]
                            split_mean = split_subset[metric].dropna().mean()
                            if not np.isnan(split_mean):
                                split_means.append(split_mean)
                        values = np.array(split_means)
                    else:
                        values = config_data[metric].dropna().values

                    if len(values) > 0:
                        rows.append({
                            "dataset": dataset,
                            "model": model,
                            "config": config,
                            "metric": metric,
                            "mean": np.mean(values),
                            "std": np.std(values, ddof=1) if len(values) > 1 else 0,
                            "n": len(values),
                        })

    return pd.DataFrame(rows)


def compute_differences(split_results: pd.DataFrame) -> pd.DataFrame:
    """Compute differences relative to MB_Shield."""

    rows = []

    for dataset in split_results["dataset"].unique():
        for model in split_results["model"].unique():
            for metric in ["accuracy", "f1", "dpd", "dpr", "eod", "eor"]:
                subset = split_results[
                    (split_results["dataset"] == dataset) &
                    (split_results["model"] == model)
                ]

                # MB_Shield baseline
                mb_data = subset[subset["config"] == "MB_Shield"]
                if mb_data.empty:
                    continue

                mb_values = mb_data[metric].dropna().values

                for config in ["Fair_Filter", "Random_Filter"]:
                    config_data = subset[subset["config"] == config]

                    if config_data.empty:
                        continue

                    if config == "Random_Filter":
                        # Average random draws within each split
                        config_values_by_split = []
                        for split_idx in config_data["split"].unique():
                            split_subset = config_data[config_data["split"] == split_idx]
                            split_mean = split_subset[metric].dropna().mean()
                            if not np.isnan(split_mean):
                                config_values_by_split.append(split_mean)
                        config_values = np.array(config_values_by_split)
                    else:
                        config_values = config_data[metric].dropna().values

                    if len(config_values) == len(mb_values) and len(mb_values) > 0:
                        diff = config_values - mb_values

                        # Paired t-test
                        try:
                            _, p_val = ttest_rel(config_values, mb_values)
                        except:
                            p_val = np.nan

                        rows.append({
                            "dataset": dataset,
                            "model": model,
                            "config": config,
                            "metric": metric,
                            "mean_diff": np.mean(diff),
                            "std_diff": np.std(diff, ddof=1) if len(diff) > 1 else 0,
                            "p_value": p_val,
                            "n_pairs": len(diff),
                        })

    return pd.DataFrame(rows)


# ==========================================
# ACCURACY ABLATION TABLE
# ==========================================

def create_accuracy_ablation_table(differences_df: pd.DataFrame) -> pd.DataFrame:
    """Create accuracy-focused ablation table."""

    rows = []

    for dataset in differences_df["dataset"].unique():
        for model in differences_df["model"].unique():
            subset = differences_df[
                (differences_df["dataset"] == dataset) &
                (differences_df["model"] == model) &
                (differences_df["metric"] == "accuracy")
            ]

            proxy_row = subset[subset["config"] == "Fair_Filter"]
            random_row = subset[subset["config"] == "Random_Filter"]

            if not proxy_row.empty and not random_row.empty:
                proxy_diff = proxy_row["mean_diff"].values[0]
                random_diff = random_row["mean_diff"].values[0]
                proxy_p = proxy_row["p_value"].values[0]
                random_p = random_row["p_value"].values[0]

                rows.append({
                    "dataset": dataset,
                    "model": model,
                    "proxy_removal_accuracy_delta": proxy_diff,
                    "random_nonproxy_removal_accuracy_delta": random_diff,
                    "proxy_minus_random_delta": proxy_diff - random_diff,
                    "proxy_p_value": proxy_p,
                    "random_p_value": random_p,
                })

    return pd.DataFrame(rows)


# ==========================================
# LATEX TABLE
# ==========================================

def save_latex_table(ablation_table: pd.DataFrame, save_path: str = "random_nonproxy_ablation_accuracy_table.tex") -> None:
    """Generate LaTeX table."""

    with open(save_path, 'w') as f:
        f.write("\\begin{table}[h]\n")
        f.write("\\centering\n")
        f.write("\\caption{Proxy removal versus random non-proxy removal. "
                "Values report mean accuracy change relative to the MB Shield configuration. "
                "Negative values indicate lower accuracy than the MB Shield reference.}\n")
        f.write("\\label{tab:random-removal-ablation}\n")
        f.write("\\begin{tabular}{llrrrr}\n")
        f.write("\\toprule\n")
        f.write("Dataset & Model & Proxy removal & Random non-proxy removal & Proxy--Random \\\\\n")
        f.write("\\midrule\n")

        for _, row in ablation_table.iterrows():
            dataset = row["dataset"]
            model = row["model"].replace("_", " ").title()
            proxy = row["proxy_removal_accuracy_delta"]
            random = row["random_nonproxy_removal_accuracy_delta"]
            diff = row["proxy_minus_random_delta"]

            f.write(f"{dataset} & {model} & {proxy:+.4f} & {random:+.4f} & {diff:+.4f} \\\\\n")

        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write("\\end{table}\n")

    print(f"✓ Saved LaTeX table: {save_path}")


# ==========================================
# PLOTTING
# ==========================================

def plot_accuracy_delta(ablation_table: pd.DataFrame, save_path: str = "random_nonproxy_ablation_accuracy_delta.png") -> None:
    """Plot accuracy delta comparison."""

    # Prepare data
    ablation_table = ablation_table.copy()
    ablation_table["dataset_model"] = ablation_table["dataset"] + " - " + ablation_table["model"].str.replace("_", " ").str.title()

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(ablation_table))
    width = 0.35

    proxy_deltas = ablation_table["proxy_removal_accuracy_delta"].values
    random_deltas = ablation_table["random_nonproxy_removal_accuracy_delta"].values

    bars1 = ax.bar(x - width/2, proxy_deltas, width, label='Proxy removal', alpha=0.8, color='#1f77b4')
    bars2 = ax.bar(x + width/2, random_deltas, width, label='Random non-proxy removal', alpha=0.8, color='#ff7f0e')

    ax.axhline(0, color='red', linestyle='-', linewidth=1, alpha=0.7)

    ax.set_ylabel("Accuracy change relative to MB_Shield", fontweight='bold', fontsize=12)
    ax.set_xlabel("Dataset - Model", fontweight='bold', fontsize=12)
    ax.set_title("Proxy Removal vs Random Non-Proxy Removal", fontweight='bold', fontsize=14)

    ax.set_xticks(x)
    ax.set_xticklabels(ablation_table["dataset_model"], rotation=45, ha='right')

    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')

    fig.tight_layout()
    fig.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"✓ Saved plot: {save_path}")
    plt.close()


# ==========================================
# REPORT
# ==========================================

def write_report(
    split_results: pd.DataFrame,
    ablation_table: pd.DataFrame,
    adult_mb: list,
    adult_proxy: list,
    acs_mb: list,
    acs_proxy: list,
    save_path: str = "random_nonproxy_ablation_report.txt"
) -> None:
    """Generate comprehensive text report."""

    with open(save_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("RANDOM NON-PROXY REMOVAL ABLATION EXPERIMENT\n")
        f.write("Markov Boundary-Based Fair Feature Selection\n")
        f.write("="*80 + "\n\n")

        f.write("OVERVIEW:\n")
        f.write("-" * 80 + "\n\n")
        f.write("This experiment compares the realized predictive cost of three feature configurations:\n\n")
        f.write("1. MB_Shield: The MB-based reference feature set (including protected attribute).\n")
        f.write("2. Fair_Filter: The MB-based feature set after removing designated proxy variables.\n")
        f.write("3. Random_Filter: The MB-based feature set after removing the same number of\n")
        f.write("   randomly selected non-proxy variables.\n\n")
        f.write("The Random_Filter configuration serves as a control for generic feature-count reduction.\n")
        f.write("If proxy removal produces a larger or more systematic accuracy reduction than random\n")
        f.write("non-proxy removal, this supports the interpretation that the observed predictive cost\n")
        f.write("is tied to removing proxy-relevant target information, consistent with the\n")
        f.write("minimality--sufficiency compromise.\n\n")

        f.write("DATASET CONFIGURATIONS:\n")
        f.write("-" * 80 + "\n\n")

        f.write("UCI Adult:\n")
        f.write(f"  MB features: {adult_mb}\n")
        f.write(f"  Proxy features: {adult_proxy}\n")
        f.write(f"  Proxy removal count: {len(adult_proxy)}\n\n")

        f.write("ACSIncome:\n")
        f.write(f"  MB features: {acs_mb}\n")
        f.write(f"  Proxy features: {acs_proxy}\n")
        f.write(f"  Proxy removal count: {len(acs_proxy)}\n\n")

        f.write("EVALUATION DESIGN:\n")
        f.write("-" * 80 + "\n\n")
        f.write("Repeated stratified holdout evaluation:\n")
        f.write("  Splits: 30\n")
        f.write("  Test size: 30%\n")
        f.write("  Stratify by: target (income)\n\n")
        f.write("Classifiers: Logistic Regression, Random Forest, Gradient Boosting\n\n")
        f.write("Metrics: Accuracy, F1-score, DPD, DPR, EOD, EOR\n\n")
        f.write("Random draws: 30 per split for Random_Filter\n\n")

        f.write("KEY RESULTS (Accuracy):\n")
        f.write("-" * 80 + "\n\n")
        f.write(ablation_table.to_string(index=False))
        f.write("\n\n")

        f.write("INTERPRETATION:\n")
        f.write("-" * 80 + "\n\n")
        f.write("The random-removal control separates the effect of proxy filtering from the generic\n")
        f.write("effect of reducing feature-set size.\n\n")
        f.write("Positive (proxy_minus_random_delta) values indicate that proxy removal caused a\n")
        f.write("larger accuracy reduction than random non-proxy removal, suggesting the cost is\n")
        f.write("tied to proxy-specific information.\n\n")
        f.write("Negative (proxy_minus_random_delta) values indicate that proxy removal caused a\n")
        f.write("smaller accuracy reduction than random non-proxy removal, which may indicate that\n")
        f.write("the selected proxies are redundant or that random feature removal was less fortunate.\n\n")
        f.write("Mixed results across datasets and models should be interpreted cautiously, as they\n")
        f.write("suggest the relationship between proxy removal and accuracy cost depends on both\n")
        f.write("the dataset structure and the model type.\n\n")

        f.write("IMPORTANT CAVEATS:\n")
        f.write("-" * 80 + "\n\n")
        f.write("• This experiment does NOT prove a causal relationship.\n")
        f.write("• The observed accuracy cost reflects the realized predictive cost on the specific\n")
        f.write("  dataset and model, not a fundamental property of the features.\n")
        f.write("• Alternative proxy or MB selections may yield different results.\n")
        f.write("• Random non-proxy removal is a heuristic control and may not fully account for\n")
        f.write("  all confounding factors.\n\n")

        f.write("="*80 + "\n")

    print(f"✓ Saved report: {save_path}")


# ==========================================
# MAIN EXECUTION
# ==========================================

if __name__ == "__main__":
    print("="*80)
    print("RANDOM NON-PROXY REMOVAL ABLATION EXPERIMENT")
    print("="*80)

    all_split_results = []
    all_ablation_tables = []

    # ==========================================
    # PART 1: UCI ADULT
    # ==========================================

    print("\n" + "="*80)
    print("PART 1: UCI ADULT")
    print("="*80)

    try:
        df_adult = load_adult()

        adult_mb_features = [
            "age",
            "education-num",
            "marital-status",
            "occupation",
            "relationship",
            "hours-per-week",
            "capital-gain",
            "capital-loss",
            "workclass",
            "race"
        ]

        adult_proxy_features = [
            "relationship",
            "marital-status",
            "occupation"
        ]

        adult_continuous = [
            "age", "capital-gain", "capital-loss", "hours-per-week", "education-num"
        ]

        split_results_adult = evaluate_dataset_ablation(
            df_adult,
            target="income",
            sensitive_attr="sex",
            mb_features=adult_mb_features,
            proxy_features=adult_proxy_features,
            continuous_columns=adult_continuous,
            dataset_name="Adult",
            n_splits=30,
            n_random_draws=30,
        )

        all_split_results.append(split_results_adult)
        print(f"\n✓ Adult ablation complete: {len(split_results_adult)} rows")

    except Exception as e:
        print(f"\n✗ Adult ablation failed: {e}")
        import traceback
        traceback.print_exc()

    # ==========================================
    # PART 2: ACSINCOME
    # ==========================================

    print("\n" + "="*80)
    print("PART 2: ACSINCOME")
    print("="*80)

    try:
        df_acs = load_acsincome(sample_size=50000)

        acs_mb_features = [
            "OCCP",
            "SCHL",
            "WKHP",
            "RELP",
            "AGEP",
            "COW",
            "RAC1P"
        ]

        acs_proxy_features = [
            "OCCP",
            "SCHL",
            "WKHP",
            "RELP"
        ]

        acs_continuous = ["AGEP", "WKHP", "SCHL"]

        split_results_acs = evaluate_dataset_ablation(
            df_acs,
            target="income",
            sensitive_attr="sex",
            mb_features=acs_mb_features,
            proxy_features=acs_proxy_features,
            continuous_columns=acs_continuous,
            dataset_name="ACSIncome",
            n_splits=30,
            n_random_draws=30,
        )

        all_split_results.append(split_results_acs)
        print(f"\n✓ ACSIncome ablation complete: {len(split_results_acs)} rows")

    except Exception as e:
        print(f"\n✗ ACSIncome ablation failed: {e}")
        import traceback
        traceback.print_exc()

    # ==========================================
    # PART 3: COMBINE AND ANALYZE
    # ==========================================

    if all_split_results:
        print("\n" + "="*80)
        print("COMBINING AND ANALYZING RESULTS")
        print("="*80)

        split_results_all = pd.concat(all_split_results, ignore_index=True)
        print(f"\nTotal split results: {len(split_results_all)}")

        # Compute summaries
        summary_df = compute_summary(split_results_all)
        print(f"Summary rows: {len(summary_df)}")

        differences_df = compute_differences(split_results_all)
        print(f"Difference rows: {len(differences_df)}")

        accuracy_ablation_table = create_accuracy_ablation_table(differences_df)
        print(f"Accuracy ablation rows: {len(accuracy_ablation_table)}")

        # ==========================================
        # PART 4: SAVE RESULTS
        # ==========================================

        print("\n" + "="*80)
        print("SAVING RESULTS")
        print("="*80)

        split_results_all.to_csv("random_nonproxy_ablation_split_results.csv", index=False)
        print("✓ Saved: random_nonproxy_ablation_split_results.csv")

        summary_df.to_csv("random_nonproxy_ablation_summary.csv", index=False)
        print("✓ Saved: random_nonproxy_ablation_summary.csv")

        differences_df.to_csv("random_nonproxy_ablation_differences.csv", index=False)
        print("✓ Saved: random_nonproxy_ablation_differences.csv")

        accuracy_ablation_table.to_csv("random_nonproxy_ablation_accuracy_table.csv", index=False)
        print("✓ Saved: random_nonproxy_ablation_accuracy_table.csv")

        # LaTeX table
        save_latex_table(accuracy_ablation_table)

        # Plot
        plot_accuracy_delta(accuracy_ablation_table)

        # Report
        write_report(
            split_results_all,
            accuracy_ablation_table,
            adult_mb_features,
            adult_proxy_features,
            acs_mb_features,
            acs_proxy_features,
        )

        # ==========================================
        # PART 5: PRINT SUMMARY
        # ==========================================

        print("\n" + "="*80)
        print("ACCURACY ABLATION TABLE")
        print("="*80)
        print(accuracy_ablation_table.to_string(index=False))

        print("\n" + "="*80)
        print("✓ ANALYSIS COMPLETE")
        print("="*80)
        print("\nGenerated files:")
        print("  • random_nonproxy_ablation_split_results.csv")
        print("  • random_nonproxy_ablation_summary.csv")
        print("  • random_nonproxy_ablation_differences.csv")
        print("  • random_nonproxy_ablation_accuracy_table.csv")
        print("  • random_nonproxy_ablation_accuracy_table.tex")
        print("  • random_nonproxy_ablation_accuracy_delta.png")
        print("  • random_nonproxy_ablation_report.txt")
        print("="*80)

    else:
        print("\n✗ No results to analyze.")

RANDOM NON-PROXY REMOVAL ABLATION EXPERIMENT

PART 1: UCI ADULT

[Loading UCI Adult Dataset]
  ✓ Downloaded successfully
  Original shape: (32561, 15)
  After dropping NaN: (30162, 15)
  Final shape: (30162, 12)

[Ablation Evaluation for Adult]
  MB features: ['age', 'education-num', 'marital-status', 'occupation', 'relationship', 'hours-per-week', 'capital-gain', 'capital-loss', 'workclass', 'race']
  Proxy features: ['relationship', 'marital-status', 'occupation']
  MB_Shield: 11 features
  Fair_Filter: 7 features
  Random_Filter: 11 features minus 3 random non-proxies

  [Split 1/30]
    MB_Shield            logistic             [1/2880] ✓
    Fair_Filter          logistic             [2/2880] ✓
    Random_Filter        logistic             draw  1 [3/2880] ✓
    Random_Filter        logistic             draw  2 [4/2880] ✓
    Random_Filter        logistic             draw  3 [5/2880] ✓
    Random_Filter        logistic             draw  4 [6/2880] ✓
    Random_Filter        logisti